# 02 - RadGraph-XL Full-Dataset Visualisation

This notebook regenerates the canonical dissertation figures for the complete
2,300-report collection. It first validates Notebook 01 outputs, then writes
aggregate figures to `outputs/full_2300/figures`. Existing files with the same
canonical names are replaced; the earlier 300-report `outputs/figures` folder
is not modified. No report text or token sequence is displayed or exported.


In [ ]:
from __future__ import annotations

import json
import os
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

MIMIC_ZIP = Path(os.environ["RADGRAPH_XL_MIMIC_ZIP"])
STANFORD_JSONL = Path(os.environ["RADGRAPH_XL_STANFORD_JSONL"])
RUN_NAME = os.getenv("RADGRAPH_XL_RUN_NAME", "full_2300")
if RUN_NAME != "full_2300":
    raise ValueError(
        "Notebook 02 regenerates formal complete-dataset figures and requires "
        "RADGRAPH_XL_RUN_NAME=full_2300"
    )

AUDIT_DIR = PROJECT_ROOT / "outputs" / RUN_NAME / "audit"
FIGURE_DIR = PROJECT_ROOT / "outputs" / RUN_NAME / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_PATH = AUDIT_DIR / "data_audit.json"
SUMMARY_PATH = AUDIT_DIR / "report_summary.csv"

assert MIMIC_ZIP.exists(), MIMIC_ZIP
assert STANFORD_JSONL.exists(), STANFORD_JSONL
assert AUDIT_PATH.exists(), "Run Notebook 01 before Notebook 02"
assert SUMMARY_PATH.exists(), "Run Notebook 01 before Notebook 02"

print(
    {
        "run_name": RUN_NAME,
        "dataset_scope": "complete 2,300-report RadGraph-XL collection",
        "audit_path": str(AUDIT_PATH),
        "figure_dir": str(FIGURE_DIR),
    }
)


In [ ]:
def read_jsonl_lines(lines, source: str) -> list[dict]:
    records = []
    for raw_line in lines:
        if raw_line.strip():
            records.append({"source": source, "record": json.loads(raw_line)})
    return records


audit = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
audited_reports = pd.read_csv(SUMMARY_PATH)
if audit.get("reports") != 2300 or len(audited_reports) != 2300:
    raise ValueError("Notebook 01 outputs do not describe the complete 2,300-report collection")

with zipfile.ZipFile(MIMIC_ZIP) as archive:
    members = [name for name in archive.namelist() if name.lower().endswith(".jsonl")]
    if len(members) != 1:
        raise ValueError(f"Expected one MIMIC JSONL member, found {members}")
    with archive.open(members[0]) as handle:
        mimic_records = read_jsonl_lines(handle, "mimic")

with STANFORD_JSONL.open("rb") as handle:
    stanford_records = read_jsonl_lines(handle, "stanford")

loaded_records = mimic_records + stanford_records
if len(loaded_records) != audit["reports"]:
    raise ValueError(
        f"Raw/audit report mismatch: raw={len(loaded_records)}, audit={audit['reports']}"
    )

print(
    {
        "mimic": len(mimic_records),
        "stanford": len(stanford_records),
        "combined": len(loaded_records),
        "audit_schema_version": audit.get("audit_schema_version"),
    }
)


In [ ]:
def span_distance(head_start: int, head_end: int, tail_start: int, tail_end: int) -> int:
    if head_end < tail_start:
        return tail_start - head_end
    if tail_end < head_start:
        return head_start - tail_end
    return 0


report_rows = []
entity_label_counts = Counter()
relation_label_counts = Counter()
relation_distances = []

for item in loaded_records:
    source = item["source"]
    record = item["record"]
    dataset = str(record["dataset"])
    tokens = [token for sentence in record["sentences"] for token in sentence]
    entities = [annotation for sentence in record["ner"] for annotation in sentence]
    relations = [annotation for sentence in record["relations"] for annotation in sentence]

    token_count = len(tokens)
    report_rows.append(
        {
            "source": source,
            "dataset": dataset,
            "doc_id": f"{dataset}::{record['doc_key']}",
            "token_count": token_count,
            "entity_count": len(entities),
            "relation_count": len(relations),
            "entities_per_100_tokens": 100 * len(entities) / token_count,
            "relations_per_100_tokens": 100 * len(relations) / token_count,
        }
    )
    entity_label_counts.update(str(annotation[2]) for annotation in entities)
    relation_label_counts.update(str(annotation[4]) for annotation in relations)
    relation_distances.extend(
        span_distance(int(hs), int(he), int(ts), int(te))
        for hs, he, ts, te, _ in relations
    )

report_df = pd.DataFrame(report_rows)
entity_df = pd.DataFrame(
    [{"label": label, "count": count} for label, count in entity_label_counts.items()]
).sort_values("count", ascending=False)
relation_df = pd.DataFrame(
    [{"label": label, "count": count} for label, count in relation_label_counts.items()]
).sort_values("count", ascending=False)

raw_ids = set(report_df["doc_id"])
audit_ids = set(audited_reports["doc_id"])
if raw_ids != audit_ids:
    raise ValueError("Notebook 02 raw report identifiers do not match Notebook 01 outputs")

print(
    report_df.groupby("source").agg(
        reports=("doc_id", "size"),
        tokens=("token_count", "sum"),
        entities=("entity_count", "sum"),
        relations=("relation_count", "sum"),
    )
)


In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
SOURCE_PALETTE = {"mimic": "#0B6E69", "stanford": "#C44E52"}
LABEL_PALETTE = ["#0B6E69", "#C44E52", "#4C72B0", "#DD8452", "#8172B2", "#55A868", "#937860"]
saved_figures = {}


def save_current_figure(name: str, caption: str, placement: str) -> None:
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight")
    saved_figures[name] = {
        "path": str(path),
        "caption": caption,
        "recommended_placement": placement,
        "dataset_scope": "complete 2,300-report RadGraph-XL collection",
        "contains_report_text": False,
    }
    plt.show()
    plt.close()


In [ ]:
source_counts = report_df["source"].value_counts().rename_axis("source").reset_index(name="reports")
plt.figure(figsize=(6, 4))
ax = sns.barplot(data=source_counts, x="source", y="reports", hue="source", palette=SOURCE_PALETTE, legend=False)
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)
plt.title("RadGraph-XL Reports by Source (n = 2,300)")
plt.xlabel("Source")
plt.ylabel("Reports")
save_current_figure(
    "source_distribution.png",
    "Distribution of the 2,300 RadGraph-XL reports by institutional source.",
    "Methodology dataset description or Appendix A",
)


In [ ]:
dataset_counts = report_df.groupby(["dataset", "source"]).size().reset_index(name="reports").sort_values("reports")
plt.figure(figsize=(10, 5))
sns.barplot(data=dataset_counts, x="reports", y="dataset", hue="source", palette=SOURCE_PALETTE)
plt.title("Reports across RadGraph-XL Source-Modality Groups (n = 2,300)")
plt.xlabel("Reports")
plt.ylabel("Dataset group")
save_current_figure(
    "modality_distribution.png",
    "Distribution of reports across the seven source-modality groups in the complete RadGraph-XL collection.",
    "Methodology dataset description",
)


In [ ]:
plt.figure(figsize=(9, 4.5))
sns.histplot(data=report_df, x="token_count", hue="source", palette=SOURCE_PALETTE, bins=40, element="step", common_norm=False)
plt.axvline(512, color="#222222", linestyle="--", linewidth=1.5, label="512 original tokens")
plt.title("Report Token-Length Distribution by Source (n = 2,300)")
plt.xlabel("Original tokens per report")
plt.ylabel("Reports")
plt.legend(
    handles=[
        Patch(facecolor=SOURCE_PALETTE["mimic"], alpha=0.35, label="MIMIC"),
        Patch(facecolor=SOURCE_PALETTE["stanford"], alpha=0.35, label="Stanford"),
        Line2D([0], [0], color="#222222", linestyle="--", label="512 original tokens"),
    ],
    title="Source / reference",
)
save_current_figure(
    "report_token_lengths.png",
    "Distribution of original-token report lengths by source; the dashed line marks 512 original tokens and is an approximate indicator rather than a WordPiece limit.",
    "Methodology long-report motivation",
)


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=entity_df, x="count", y="label", hue="label", palette=LABEL_PALETTE[:len(entity_df)], legend=False)
plt.title("Entity Label Distribution in RadGraph-XL (n = 229,771)")
plt.xlabel("Entity annotations")
plt.ylabel("Entity label")
save_current_figure(
    "entity_label_distribution.png",
    "Distribution of the six assertion-aware entity labels in the complete RadGraph-XL collection.",
    "Methodology dataset description or Appendix A",
)


In [ ]:
plt.figure(figsize=(7, 4.5))
sns.barplot(data=relation_df, x="label", y="count", hue="label", palette=LABEL_PALETTE[:len(relation_df)], legend=False)
plt.title("Relation Label Distribution in RadGraph-XL (n = 180,106)")
plt.xlabel("Relation label")
plt.ylabel("Relation annotations")
save_current_figure(
    "relation_label_distribution.png",
    "Distribution of positive relation annotations in the complete RadGraph-XL collection.",
    "Methodology dataset description",
)


In [ ]:
plt.figure(figsize=(8, 5.5))
sns.scatterplot(data=report_df, x="entity_count", y="relation_count", hue="source", palette=SOURCE_PALETTE, alpha=0.55, s=26)
plt.title("Entity and Relation Counts per Report (n = 2,300)")
plt.xlabel("Entities per report")
plt.ylabel("Relations per report")
save_current_figure(
    "report_annotation_counts.png",
    "Relationship between entity and relation annotation counts at report level, separated by source.",
    "Appendix A dataset statistics",
)


In [ ]:
distance_cap = 100
distance_series = pd.Series(relation_distances, name="distance")
plt.figure(figsize=(9, 4.5))
sns.histplot(distance_series[distance_series <= distance_cap], bins=40, color="#4C72B0")
plt.yscale("log")
plt.title(f"Gold Relation Span Distance (display capped at {distance_cap} tokens)")
plt.xlabel("Original-token gap between entity spans")
plt.ylabel("Relations (log scale)")
save_current_figure(
    "relation_distance_distribution.png",
    "Distribution of directed gold-relation span gaps up to 100 original tokens; the logarithmic y-axis reveals the sparse long-distance tail.",
    "Methodology relation-candidate design or Appendix A",
)


In [ ]:
entity_grid = entity_df.copy()
entity_grid[["category", "assertion"]] = entity_grid["label"].str.split("::", n=1, expand=True)
pivot = entity_grid.pivot_table(index="category", columns="assertion", values="count", aggfunc="sum", fill_value=0)
plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt=",", cmap="YlGnBu")
plt.title("Entity Category and Assertion Distribution")
plt.xlabel("Assertion")
plt.ylabel("Entity category")
save_current_figure(
    "entity_category_assertion_distribution.png",
    "Cross-tabulation of entity category and assertion status in the complete RadGraph-XL collection.",
    "Appendix A dataset statistics",
)


In [ ]:
manifest_path = FIGURE_DIR / "dataset_figure_manifest.json"
manifest = {
    "generated_by": "notebooks/02_data_visualization.ipynb",
    "run_name": RUN_NAME,
    "canonical_output": True,
    "dataset_scope": "complete 2,300-report RadGraph-XL collection",
    "contains_report_text": False,
    "figures": saved_figures,
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved canonical aggregate figures:")
for name, metadata in saved_figures.items():
    print(f" - {name}: {metadata['path']}")
print(" - manifest:", manifest_path)
